# 01 · OLS in statsmodels

**Module:** 4 · **Time:** 45 minutes · **Prerequisites:** 00 OLS by hand; pandas fundamentals

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/04_econometrics/01_ols_statsmodels.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F04_econometrics%2F01_ols_statsmodels.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** Badges target `main` after merge; this is a private repository and draft PR.

**Completed solution edition.**

## Learning goals

- **F**it a reproducible multiple regression using a formula.
- **I**nterpret coefficients, fitted values, and model fit in stated units.
- **D**istinguish statistical association from causal identification.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(512)
n_obs = 150
cluster_id = np.repeat(np.arange(30), 5)
schooling = rng.integers(10, 19, size=n_obs)
experience = rng.integers(0, 25, size=n_obs)
cluster_shocks = rng.normal(0, 2, size=30)
person_shocks = rng.normal(0, 3, size=n_obs)

In [ ]:
wage_index = (5 + 1.6 * schooling + 0.5 * experience
              + cluster_shocks[cluster_id] + person_shocks)
survey = pd.DataFrame({"wage_index": wage_index,
                       "schooling": schooling, "experience": experience,
                       "cluster_id": cluster_id})
print("Fictional workers:", len(survey))
print(survey.head(3))

## Why this matters

Once we understand OLS mathematically, a tested package helps keep fitted coefficients, diagnostics and predictions together. These wages, schooling levels and experience years are **simulated**, not observations from real workers. The model is descriptive unless an identification strategy supports causal interpretation.

## Estimate a model with an intercept

statsmodels' formula interface includes an intercept by default. `wage_index ~ schooling + experience` asks for the conditional linear association with both regressors. OLS reports conventional standard errors until you request an alternative covariance estimator.

In [ ]:
baseline = smf.ols("wage_index ~ schooling + experience",
                   data=survey).fit()
print(baseline.params.round(3))
print("R-squared:", round(baseline.rsquared, 3))

## Read coefficients and uncertainty

The schooling coefficient represents the change in the **fictional wage index** associated with one additional year of schooling, holding experience fixed within this fitted linear model. A conventional confidence interval is conditional on modeling assumptions.

In [ ]:
print("Schooling coefficient:", baseline.params["schooling"])
print("Conventional standard error:", baseline.bse["schooling"])
print("Conventional 95% CI:")
print(baseline.conf_int().loc["schooling"])

## Predict and inspect residuals

Use `predict` for fitted values at observed or new regressor levels; make sure the new data uses the same column names and units.

In [ ]:
new_workers = pd.DataFrame({"schooling": [12, 16],
                            "experience": [4, 10]})
predictions = baseline.predict(new_workers)
print("Predicted fictional wage indexes:", predictions.to_list())
print("Residual mean:", baseline.resid.mean())

## Check formula and matrix estimates agree

Adding a column of ones to a NumPy design matrix makes the intercept explicit. The numerical least-squares solution agrees with the formula fit on this fully observed sample.

In [ ]:
x_design = np.column_stack((np.ones(n_obs), schooling, experience))
matrix_beta = np.linalg.lstsq(x_design, wage_index, rcond=None)[0]
assert np.allclose(matrix_beta, baseline.params.to_numpy())
print("Formula and matrix OLS agree")

## If you know Stata/R

| Task | Python | Stata | R |
| --- | --- | --- | --- |
| OLS | `smf.ols('wage_index ~ schooling + experience',data=d).fit()` | `reg wage_index schooling experience` | `lm(wage_index ~ schooling + experience,data=d)` |
| Coefficients | `result.params` | `matrix list e(b)` | `coef(result)` |
| Prediction | `result.predict(new_data)` | `predict yhat` | `predict(result,newdata=...)` |

The next lesson teaches heteroskedastic-robust and cluster-robust inference.

## Try it

Exercises begin with `None` in the learner notebook. Checks offer hints rather than raising errors, while completed solutions must pass every check.

### ★ Exercise 1 — Fit a simple regression

Fit `first_fit` for the synthetic wage index as a function of schooling alone, using `smf.ols` and `.fit()`.

In [ ]:
first_fit = smf.ols("wage_index ~ schooling", data=survey).fit()

In [ ]:
if (hasattr(first_fit, "params") and
    "schooling" in first_fit.params.index and
    int(first_fit.nobs) == len(survey)):
    print("✓ Simple OLS estimated")
else:
    print("Hint: call smf.ols(formula, data=survey).fit().")

### ★★ Exercise 2 — Extract a coefficient

Set `schooling_beta` to the coefficient on schooling from `baseline`.

In [ ]:
schooling_beta = float(baseline.params["schooling"])

In [ ]:
if (isinstance(schooling_beta, (int, float, np.number)) and
    np.isclose(schooling_beta, baseline.params["schooling"])):
    print("✓ Correct coefficient extracted")
else:
    print("Hint: select the 'schooling' entry of baseline.params.")

### ★★ Exercise 3 — Predict three outcomes

Create `three_predictions` for the first three survey rows using `baseline.predict`.

In [ ]:
three_predictions = baseline.predict(survey.head(3))

In [ ]:
expected_predictions = baseline.predict(survey.head(3))
if (isinstance(three_predictions, pd.Series) and
    np.allclose(three_predictions.to_numpy(), expected_predictions.to_numpy())):
    print("✓ Predictions correct")
else:
    print("Hint: pass the first three survey rows to baseline.predict().")

### ★★★ Exercise 4 — Check the OLS residual mean

Store the sample mean of `baseline.resid` in `residual_mean`.

In [ ]:
residual_mean = float(baseline.resid.mean())

In [ ]:
if isinstance(residual_mean, (int, float, np.number)) and np.isclose(
    residual_mean, 0.0, atol=1e-10
):
    print("✓ Residual mean is approximately zero")
else:
    print("Hint: take the mean of baseline.resid; OLS includes an intercept.")

## Common errors

- `NameError: smf is not defined` — import the formula API before estimating a model.
- `PatsyError` or formula evaluation errors often mean a column was misspelled.
- Without an intercept, regression coefficients and R-squared have different interpretations.
- A statistically significant coefficient does not by itself justify a causal claim.

## Recap

- The formula API organizes estimators, coefficients and predictions.
- An OLS coefficient is an adjusted linear association in stated units.
- Exogeneity and identification are separate from fitting a regression.

## Next steps

Proceed to [Standard errors and tests](02_standard_errors_and_tests.ipynb). Read the [official statsmodels OLS documentation](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md) with this lesson path.